In [16]:
import numpy as np

def pad_truncate_2d(arr, maxlen):
    """
    仅对第 0 维（batch）截断或补零，第 1 维保持不变
    """
    arr = np.asarray(arr)
    B = arr.shape[0]
    if B >= maxlen:                      # 截断
        return arr[:maxlen]
    else:                                # 补零
        pad_width = [(0, maxlen - B), (0, 0)]
        return np.pad(arr, pad_width, 'constant')


# 演示
a = np.ones((3, 4))
print(pad_truncate_2d(a, 5))   # (5, 4) 后面补 0
print()
print(pad_truncate_2d(a, 2))   # (2, 4) 前面截断

[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

[[1. 1. 1. 1.]
 [1. 1. 1. 1.]]


In [10]:
from contrib.state_space import *

In [2]:
space = StateSpace()

In [3]:
space

StateSpace(6)

In [5]:
gspace = space.to_gym_space()

In [8]:
gspace.sample()

(array([ 1.2168516 , -0.9984723 ,  1.2148839 , -0.064732  ,  0.59260595,
         0.10890837], dtype=float32),
 array([ 1.1936984, -1.1225674, -1.7435038,  1.5018108, -0.9095504,
         1.0704478], dtype=float32),
 array([-0.26611176,  0.26204994,  0.57032245, -0.929515  ,  1.0233603 ,
        -0.94037193], dtype=float32))

In [11]:
# demo_state_space.py
import numpy as np
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag

# 1. 造一个极简电路：q0-H-┐
#                      q1---CX
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

# 2. 转成 DAG
dag = circuit_to_dag(qc)

# 3. 伪造距离矩阵（2×2 即可）
dist = np.array([[0, 1],
                 [1, 0]], dtype=int)


    === unrouted ===
    [[ 0  1  -1  0  0  0]   # H  q0→0, 无真正2-q → dist=-1
     [ 1  1  1  1  0  1]]  # CX q0→0,q1→1, dist=1



In [12]:

# 4. 实例化编码器
encoder = StateSpace()

# 5. 两种场景快速测试
print("=== unrouted ===")
seq_un = encoder.encode_dag(dag,
                            routed_status=RoutedStatus.UNROUNTED,
                            current_mapping={dag.qubits[0]: 0, dag.qubits[1]: 1},
                            distance_matrix=dist)
print(seq_un)

=== unrouted ===
[[ 0  1 -1  0  0  0]
 [ 1  1  1  1  0  1]]


In [13]:

print("\n=== routed ===")
seq_ro = encoder.encode_dag(dag,
                            routed_status=RoutedStatus.ROUTED,
                            current_mapping=None,
                            distance_matrix=None)  # 不会被用到
print(seq_ro)


=== routed ===
[[ 0 -1 -1  0  0  0]
 [ 1 -1 -1  1  0  1]]


    === routed ===
    [[ 0 -1 -1  0  0  0]   # H  routed → dist=-1
     [ 1 -1 -1  1  0  1]]  # CX routed → dist=-1